# 📖 Notebook 2: Common Web Vulnerabilities

In this notebook, we'll **attack** our Flask app with the four most common web vulnerabilities, then **fix** each one.

These vulnerabilities are in the [OWASP Top 10](https://owasp.org/www-project-top-ten/) — the industry-standard list of critical web security risks.

## Learning Objectives

By the end of this notebook, you'll understand:
- How SQL injection works and how to prevent it
- How Cross-Site Scripting (XSS) works and how to prevent it
- Why Server-Side Template Injection (SSTI) hides behind XSS, and why HTML-escaping does **not** fix it
- How Cross-Site Request Forgery (CSRF) works and how to prevent it
- How Server-Side Request Forgery (SSRF) works and how to prevent it
- Why each fix works at a technical level

## 🛠️ Setup

Make sure Docker is running:

```bash
cd 08-enterprise/security-review
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import requests
import json

BASE_URL = "http://localhost:5001"

# Verify the Flask app is running
try:
    resp = requests.get(f"{BASE_URL}/health")
    print(f"✅ Flask app is running: {resp.json()}")
except requests.ConnectionError:
    print("❌ Flask app is not running. Run: docker compose up -d")

---

## 🚨 Vulnerability 1: SQL Injection

**OWASP Category**: A03:2021 — Injection

### What is SQL Injection?

SQL injection happens when user input is **directly inserted into a SQL query** without sanitization. The attacker sends input that changes the query's meaning.

### How our vulnerable code works

```python
# 🚨 VULNERABLE — string concatenation
sql = f"SELECT id, name, price FROM products WHERE name LIKE '%{query}%'"
cur.execute(sql)
```

If the user searches for `laptop`, the query becomes:
```sql
SELECT id, name, price FROM products WHERE name LIKE '%laptop%'
```

But if the user searches for `' OR '1'='1' --`, the query becomes:
```sql
SELECT id, name, price FROM products WHERE name LIKE '%' OR '1'='1' --%'
```

The `OR '1'='1'` is always true, so it returns **ALL products**. The `--` comments out the rest of the query.

In [ ]:
# === ATTACK 1A: SQL Injection — Extract ALL products ===

print("Normal search for 'Laptop':")
resp = requests.get(f"{BASE_URL}/api/products/search", params={"q": "Laptop"})
normal = resp.json()
print(f"  Found {len(normal)} products")
for p in normal[:3]:
    print(f"  - {p['name']} (${p['price']})")

print("\n" + "=" * 60)
print("\n🚨 SQL Injection attack — get ALL products:")
payload = "' OR '1'='1' --"
print(f"  Payload: {payload}")
resp = requests.get(f"{BASE_URL}/api/products/search", params={"q": payload})
injected = resp.json()
print(f"  Found {len(injected)} products (ALL of them!)")
for p in injected[:5]:
    print(f"  - {p['name']} (${p['price']})")

print(f"\n⚠️  The attacker got all {len(injected)} products instead of the "
      f"{len(normal)} that matched the search term.")

assert len(injected) > len(normal), (
    f"injection returned {len(injected)} rows and the honest search returned "
    f"{len(normal)} — the vulnerable endpoint is no longer vulnerable, so this "
    f"notebook is demonstrating nothing"
)

In [ ]:
# === ATTACK 1B: SQL Injection — Extract data from OTHER tables (UNION attack) ===

# The UNION attack lets us combine results from different tables
# Our products query returns 3 columns (id, name, price)
# So we UNION with a query that also returns 3 columns from the users table

union_payload = "' UNION SELECT id, username || ':' || email, 0 FROM users --"
print(f"🚨 UNION SQL Injection — extracting user data:")
print(f"  Payload: {union_payload}")

resp = requests.get(f"{BASE_URL}/api/products/search", params={"q": union_payload})
results = resp.json()

leaked = [r for r in results if ":" in r["name"] and "@" in r["name"]]

print(f"\n  Results ({len(results)} rows, {len(leaked)} of them from `users`):")
for r in leaked:
    print(f"  🔴 LEAKED USER DATA: {r['name']}")

print("\n⚠️  The attacker extracted usernames and emails from the users table!")
print("In a real app, they could also extract password hashes, API keys, etc.")

assert len(leaked) >= 4, (
    f"expected the UNION to pull the 4 seeded users out of the `users` table, "
    f"got {len(leaked)} — either the seed data or the injection changed"
)

In [ ]:
# === FIX: Parameterized queries prevent SQL injection ===

print("✅ Testing the SAFE endpoint with the same attack payloads:\n")

# Try the OR 1=1 attack
payload1 = "' OR '1'='1' --"
safe1 = requests.get(f"{BASE_URL}/api/products/search/safe", params={"q": payload1}).json()
print(f"Payload: {payload1}")
print(f"  Results: {len(safe1)} products (the payload is treated as literal text)")

# Try the UNION attack
payload2 = "' UNION SELECT id, username || ':' || email, 0 FROM users --"
safe2 = requests.get(f"{BASE_URL}/api/products/search/safe", params={"q": payload2}).json()
print(f"\nPayload: {payload2}")
print(f"  Results: {len(safe2)} products (UNION is treated as literal text)")

# Normal search still works
safe_normal = requests.get(f"{BASE_URL}/api/products/search/safe", params={"q": "Laptop"}).json()
print(f"\nNormal search 'Laptop': {len(safe_normal)} products")

print("\n✅ Parameterized queries treat the ENTIRE input as data, not as SQL code.")
print("The database driver escapes special characters automatically.")

# A fix has to do two things: stop the attack AND keep the feature working.
# Assert both — a `WHERE false` would pass the first test and fail the second.
assert safe1 == [] and safe2 == [], (
    f"the safe endpoint returned rows for an injection payload "
    f"({len(safe1)}, {len(safe2)}) — it is not actually parameterized"
)
assert len(safe_normal) >= 1, (
    "the safe endpoint returns nothing for a legitimate search — the 'fix' broke "
    "the feature, which is not a fix"
)
assert not any(":" in p["name"] and "@" in p["name"] for p in safe2), (
    "user rows leaked through the parameterized endpoint"
)

### Why Parameterized Queries Work

```python
# ✅ SAFE — parameterized query
sql = "SELECT id, name, price FROM products WHERE name LIKE %s"
cur.execute(sql, (f"%{query}%",))
```

With parameterized queries:
1. The SQL structure is sent to the database **first**
2. The parameters are sent **separately**
3. The database treats parameters as **data only**, never as SQL code
4. Special characters like `'`, `--`, `UNION` are automatically escaped

**Rule: NEVER build SQL strings with f-strings, `.format()`, or `+` concatenation.**

---

## 🚨 Vulnerability 2: Cross-Site Scripting (XSS) — and its nastier cousin, SSTI

**OWASP Category**: A03:2021 — Injection

### What is XSS?

XSS happens when an attacker injects **JavaScript code** into a web page that other users view. The injected code runs in the victim's browser with full access to their session.

### Types of XSS

| Type | How It Works | Persistence |
|------|-------------|-------------|
| **Stored XSS** | Malicious script saved in database, served to all users | Permanent until removed |
| **Reflected XSS** | Malicious script in URL parameter, reflected in response | One-time per click |
| **DOM-based XSS** | Client-side JavaScript processes untrusted data | Client-only |

Our demo uses **Stored XSS** — the attacker saves a malicious comment in the database.

### The bug hiding behind the bug

Our vulnerable endpoint does something worse than sloppy string building. It calls:

```python
html = "<h2>Product Comments</h2>"
for username, content, created_at in comments:
    html += f"<div><b>{username}</b>: {content} ...</div>"
return render_template_string(html)   # 🚨
```

`render_template_string` **compiles its argument as a Jinja template**. The user's
comment is now part of the *template source*, not part of the data. That is
**Server-Side Template Injection (SSTI)**, and it is a different, more severe bug
than XSS: XSS runs in the victim's browser, SSTI runs **on your server**, and Jinja
expressions are a well-trodden path from there to reading files and running commands.

Keep the distinction in mind, because it decides what counts as a fix:

| | Runs where | Escaping `< > & "` fixes it? |
|---|---|---|
| **XSS** | victim's browser | ✅ yes |
| **SSTI** | your server | ❌ **no** — `escape()` does not touch `{{` or `{%` |

We'll attack both, and then check that the "safe" endpoint really closes both.

In [ ]:
# === ATTACK 2: Stored XSS + SSTI — inject via a comment ===

xss_payload  = '<script>alert("XSS! I could steal your cookies: " + document.cookie)</script>'
# 7 * 191 * 1000 == 1337000. If the number comes back instead of the braces, the
# server evaluated our input — template injection, not just unescaped output.
# Seven digits, deliberately: the page also prints each comment's timestamp, and
# the longest run of digits in one of those is the 6-digit microseconds field, so
# a 7-digit marker cannot show up by accident.
ssti_payload = '{{7*191*1000}}'
SSTI_MARKER = '1337000'

print("🚨 Injecting payloads as product comments...")
for label, payload in [("XSS ", xss_payload), ("SSTI", ssti_payload)]:
    resp = requests.post(f"{BASE_URL}/api/comments", json={
        "user_id": 2,
        "product_id": 1,
        "content": payload,
    })
    assert resp.status_code == 201, f"could not store the {label} comment: {resp.text}"
    print(f"  {label}: {payload[:60]}  -> comment id {resp.json()['id']}")

# Now fetch the comments page (vulnerable version)
print("\n📄 Fetching the VULNERABLE comments page...")
resp = requests.get(f"{BASE_URL}/comments/1")
html = resp.text

print(f"\n  Response HTML (first 500 chars):")
print(f"  {html[:500]}")

print("\n--- What came back ---")
if "<script>" in html:
    print("⚠️  The raw <script> tag is in the HTML. In a browser this JavaScript")
    print("   executes and can ship the user's session cookie to the attacker.")
if SSTI_MARKER in html and ssti_payload not in html:
    print(f"🔥 And {ssti_payload} came back as {SSTI_MARKER} — the SERVER evaluated it.")
    print("   That is SSTI. The attacker is now running expressions inside our")
    print("   Flask process, which is a foothold, not a popup.")

assert "<script>" in html, (
    "no raw <script> in the vulnerable page — the stored-XSS demo is not "
    "reproducing its own lesson"
)
assert SSTI_MARKER in html, (
    "the template payload was not evaluated by the vulnerable page — the SSTI demo is not "
    "reproducing its own lesson"
)

In [ ]:
# === FIX: escape the output AND keep user data out of the template compiler ===

print("✅ Fetching the SAFE comments page...")
resp = requests.get(f"{BASE_URL}/comments/1/safe")
html = resp.text

print(f"\n  Response HTML (first 500 chars):")
print(f"  {html[:500]}")

print("\n--- Check 1: is the XSS neutralised? ---")
if "&lt;script&gt;" in html:
    print("✅ The <script> tag is HTML-escaped to &lt;script&gt; — the browser")
    print("   shows it as text instead of executing it.")

print("\n--- Check 2: is the SSTI neutralised? ---")
if ssti_payload in html:
    print(f"✅ {ssti_payload} came back verbatim — the server did not evaluate it.")

assert "<script>" not in html, "raw <script> survived into the 'safe' page — XSS is NOT fixed"
assert "&lt;script&gt;" in html, "the payload vanished entirely; check the comment was stored"
assert SSTI_MARKER not in html, (
    "the 'safe' page evaluated the template payload — it is still passing user data to the "
    "template compiler. escape() does not stop SSTI; this is the exact failure "
    "mode where a 'fixed' example is still exploitable"
)
assert ssti_payload in html, "expected the braces to come back as literal text"

print("\n💡 Two separate things had to change:")
print("   1. escape() converts  <  >  &  \"  '  into entities  -> stops XSS")
print("   2. the finished HTML is returned with make_response(), NOT passed to")
print("      render_template_string()                            -> stops SSTI")
print("\nIf you only did (1) — which is what the textbook 'escape your output'")
print("advice sounds like — the page would still evaluate {{ ... }} server-side.")
print("The general rule is stronger than 'escape': user data is a template")
print("*variable*, never part of the template *text*.")

### Additional XSS Defenses

HTML escaping is the minimum. Production apps also use:

| Defense | What It Does |
|---------|-------------|
| **Content-Security-Policy header** | Tells browser to only run scripts from trusted sources |
| **HttpOnly cookies** | Prevents JavaScript from reading session cookies |
| **Template auto-escaping** | Frameworks like Jinja2 can escape by default |
| **Input validation** | Reject input that looks like HTML/JavaScript |

On that third row: auto-escaping is the reason a normal Flask app is not full of
XSS holes. `render_template("comments.html", comments=comments)` puts the user data
in as a **variable**, and Jinja escapes variables automatically. Our vulnerable
endpoint bypassed that protection by building the template out of user data — which
is why the safest habit is not "remember to escape" but **"never let user input
reach a template compiler, a SQL parser, or a shell."** It's the same rule three
times: keep code and data in separate arguments.

---

## 🚨 Vulnerability 3: Cross-Site Request Forgery (CSRF)

**OWASP Category**: Broken Access Control / Insecure Design

### What is CSRF?

CSRF tricks a logged-in user into making a request they didn't intend. Imagine:

1. Alice is logged into her bank at `bank.com`
2. Alice visits `evil.com` (a malicious site)
3. `evil.com` contains a hidden form that submits to `bank.com/transfer`
4. Because Alice's browser has the bank's session cookie, the transfer goes through
5. Alice just transferred money to the attacker — without clicking anything!

```html
<!-- evil.com's hidden form -->
<form action="http://bank.com/api/transfer" method="POST" id="csrf-form">
    <input type="hidden" name="from_user" value="alice">
    <input type="hidden" name="to_user" value="attacker">
    <input type="hidden" name="amount" value="10000">
</form>
<script>document.getElementById('csrf-form').submit();</script>
```

### The detail that decides whether a CSRF demo is honest

A cross-site `<form>` can only send three content types:
`application/x-www-form-urlencoded`, `multipart/form-data`, `text/plain`. Those are
the **"simple requests"** that browsers let out without a CORS preflight. If the
attacker's page instead used `fetch()` with `Content-Type: application/json`, the
browser would send an `OPTIONS` preflight first, our server would not answer it with
permissive CORS headers, and **the browser would never send the real request**.

So the attack below sends a *form-encoded* body, because that is the only shape a
real browser CSRF can take. A demo that posts JSON is testing something the browser
would have blocked, and would make an endpoint look vulnerable in a way it isn't.

> ⚠️ **Honest caveat about this particular app.** `/api/transfer` has no login and
> reads no session, so strictly speaking there is no session here to *forge* — the
> endpoint is simply unauthenticated, which is the separate (and worse) `[E]` threat
> from notebook 1. We attach a session cookie below to make the request the right
> *shape*, and the CSRF token check we add in the fix is the control you would need
> once real authentication exists. Don't take "the transfer succeeded" as proof of
> CSRF on its own — CSRF requires an ambient credential the browser attaches for you.

In [ ]:
# === ATTACK 3: CSRF — transfer funds without user consent ===

print("🚨 CSRF Attack Simulation")
print("=" * 60)
print("\nScenario: Alice is logged in to our app. She opens evil.com, whose page")
print("auto-submits a hidden form at our /api/transfer endpoint.\n")

# A browser attaches the victim's cookies to a cross-site form POST automatically.
victim_browser = requests.Session()
victim_browser.cookies.set("session_id", "alices-logged-in-session")

resp = victim_browser.post(
    f"{BASE_URL}/api/transfer",
    # form-encoded, NOT json — this is the only body a cross-site <form> can send
    data={"from_user": "alice", "to_user": "attacker", "amount": "10000"},
    headers={
        "Origin": "http://evil.com",              # browsers set this on cross-site POSTs
        "Referer": "http://evil.com/free-stuff",
    },
)

print(f"Request Content-Type: application/x-www-form-urlencoded")
print(f"Request Origin:       http://evil.com")
print(f"Cookies sent:         session_id=alices-logged-in-session")
print(f"\nResponse status: {resp.status_code}")
print(f"Response body:   {resp.json()}")
print("\n⚠️  Transfer succeeded! The server didn't check:")
print("   1. Where the request came from (Origin header said evil.com)")
print("   2. Whether a CSRF token was included")
print("   3. Whether the user actually initiated this action")

assert resp.status_code == 200, (
    f"expected the unprotected endpoint to accept a cross-site form POST, got "
    f"{resp.status_code}. A 415 here means the endpoint only parses JSON bodies — "
    f"which would make this demo untestable by a real browser form"
)
assert resp.json()["status"] == "transferred", resp.json()

In [ ]:
# === FIX: CSRF protection with Origin check and CSRF token ===

print("✅ Testing the SAFE transfer endpoint\n")

body = {"from_user": "alice", "to_user": "attacker", "amount": "10000"}
blocked = {}

# Attack 1: the real CSRF shape — cross-site form POST from evil.com
r1 = victim_browser.post(f"{BASE_URL}/api/transfer/safe", data=body,
                         headers={"Origin": "http://evil.com"})
blocked["cross-site Origin"] = r1
print(f"  Origin: evil.com                 → {r1.status_code} — {r1.json()}")

# Attack 2: attacker spoofs the Origin header, but has no CSRF token.
# (A browser will not let a page lie about Origin; this models a non-browser client.)
r2 = victim_browser.post(f"{BASE_URL}/api/transfer/safe", data=body,
                         headers={"Origin": "http://localhost:5001"})
blocked["right Origin, no token"] = r2
print(f"  Origin: ours, no CSRF token      → {r2.status_code} — {r2.json()}")

# Attack 3: a guessed CSRF token.
r3 = victim_browser.post(f"{BASE_URL}/api/transfer/safe", data=body,
                         headers={"Origin": "http://localhost:5001",
                                  "X-CSRF-Token": "guessed-token-123"})
blocked["right Origin, wrong token"] = r3
print(f"  Origin: ours, wrong CSRF token   → {r3.status_code} — {r3.json()}")

print("\n✅ All three blocked. The safe endpoint requires:")
print("   1. Origin header matches the allowed list")
print("   2. A CSRF token in the X-CSRF-Token header")
print("   3. That token matches the one stored in Redis for THIS session")

for label, r in blocked.items():
    assert r.status_code == 403, (
        f"'{label}' got {r.status_code}, expected 403 — the CSRF protection has a hole"
    )

print("\n💡 Why the token is the load-bearing part, not the Origin check:")
print("   evil.com's page cannot read our Redis-backed token, and cannot forge the")
print("   Origin header — the browser sets it. But Origin is missing on some")
print("   legitimate requests and spoofable by any non-browser client, so it is a")
print("   cheap extra layer, not the control. The token is the control.")

### CSRF Prevention Best Practices

| Defense | How It Works |
|---------|-------------|
| **CSRF Token** | Server generates a random token, stores in session. Client must send it back. |
| **SameSite Cookies** | `Set-Cookie: session=abc; SameSite=Strict` — browser won't send cookie from other sites |
| **Origin/Referer Check** | Server rejects requests from unexpected origins |
| **Custom Headers** | Require `X-Requested-With: XMLHttpRequest` (browsers block this cross-origin) |

---

## 🚨 Vulnerability 4: Server-Side Request Forgery (SSRF)

**OWASP Category**: A10:2021 — Server-Side Request Forgery

### What is SSRF?

SSRF happens when an attacker makes **your server** send HTTP requests to internal resources that should be unreachable from the internet.

```
Internet          Your Server          Internal Network
┌────────┐       ┌──────────┐         ┌──────────────┐
│Attacker│──────▶│ Flask App│────────▶│ Redis :6379  │
│        │       │          │         │ Postgres:5432│
│ "fetch  │       │/api/     │         │ AWS metadata │
│  this   │       │fetch-url │         │ 169.254.169. │
│  URL"   │       │          │         │ 254          │
└────────┘       └──────────┘         └──────────────┘
     ❌                                     ❌
  Can't reach                          Attacker reaches
  directly                             via YOUR server
```

The most famous SSRF attack was the **2019 Capital One breach**: an attacker used SSRF to access AWS metadata credentials, stealing data on 100 million customers.

In [ ]:
# === ATTACK 4A: SSRF — Access internal services ===

print("🚨 SSRF Attack — Accessing Internal Services")
print("=" * 70)

# IMPORTANT: these are docker-compose SERVICE NAMES, not localhost.
# The Flask app runs inside its own container, so "localhost" there means the
# Flask container itself -- NOT your laptop, and NOT the redis/adminer
# containers. On the compose network those are reachable as `redis` and
# `adminer`. Getting this wrong is why an SSRF demo can look like it "works"
# while actually proving nothing: every probe just gets connection-refused.
internal_targets = [
    ("http://localhost:5001/health", "the Flask app talking to itself"),
    ("http://adminer:8080/",         "Adminer — the database admin GUI"),
    ("http://redis:6379/",           "Redis — not an HTTP server, but is the port open?"),
]

# First: can *we* reach those names from this notebook? We are outside the
# compose network, so we should not be able to.
print("\nStep 1 — what can this notebook reach directly?")
for url, _ in internal_targets[1:]:
    try:
        requests.get(url, timeout=3)
        print(f"  {url:28s} reachable from here")
    except Exception as e:
        print(f"  {url:28s} ❌ {type(e).__name__} — we cannot resolve or reach it")

print("\nStep 2 — now ask the SERVER to fetch the same URLs for us:")
reached = {}
for url, description in internal_targets:
    resp = requests.get(f"{BASE_URL}/api/fetch-url", params={"url": url}, timeout=20)
    data = resp.json()
    if "status_code" in data:
        verdict = f"🔴 REACHED — HTTP {data['status_code']}"
        preview = data.get("content", "")[:80].replace("\n", " ")
        reached[url] = True
    elif "refused" in str(data.get("error", "")).lower():
        verdict = "⚪ nothing listening on that port"
        preview = data.get("error", "")[:80]
        reached[url] = False
    else:
        # A protocol error is not a refusal: something IS listening, it just
        # doesn't speak HTTP. That is a working port scanner.
        verdict = "🟠 PORT OPEN — answered, but not with HTTP"
        preview = str(data.get("error", ""))[:80]
        reached[url] = True

    print(f"\n  {description}")
    print(f"    {url}")
    print(f"    {verdict}")
    print(f"    {preview}...")

print("\n⚠️  The attacker used our server as a proxy into a network they cannot")
print("   touch directly. In AWS the next request is")
print("   http://169.254.169.254/latest/meta-data/iam/security-credentials/")
print("   which hands over IAM credentials — that is the Capital One breach.")

assert reached.get("http://adminer:8080/"), (
    "the server could not reach http://adminer:8080/ — either adminer is not "
    "running (docker compose up -d) or the SSRF hole is closed; either way this "
    "cell is no longer demonstrating SSRF"
)
assert reached.get("http://redis:6379/"), (
    "the server could not touch redis:6379 — the port-scan half of the demo is dead"
)

In [ ]:
# === FIX: URL validation, allowlisting, and IP blocking ===

print("✅ Testing the SAFE fetch-url endpoint\n")

# 400/403 mean *our* checks refused. 500 means our checks let it through and the
# fetch itself failed (e.g. no internet) — a very different outcome.
BLOCKED_BY_US = (400, 403)

test_cases = [
    # (url, description, expect)
    ("http://localhost:5001/health", "HTTP instead of HTTPS",  "blocked"),
    ("https://localhost:5001/health", "resolves to 127.0.0.1", "blocked"),
    ("http://adminer:8080/",          "the SSRF target from above", "blocked"),
    ("https://evil.com/steal-data",   "not in the allowlist",  "blocked"),
    ("https://httpbin.org/get",       "allowlisted domain",    "allowed"),
]

results = {}
for url, description, expect in test_cases:
    resp = requests.get(f"{BASE_URL}/api/fetch-url/safe", params={"url": url}, timeout=20)
    results[url] = resp.status_code
    refused = resp.status_code in BLOCKED_BY_US
    ok = refused if expect == "blocked" else not refused
    print(f"  {'✅' if ok else '❌'} {description}")
    print(f"     URL: {url}")
    print(f"     Response: {resp.status_code} — {resp.json().get('error', 'OK')}")
    print()

print("The safe endpoint enforces:")
print("  1. HTTPS only (no HTTP)")
print("  2. Block private/internal IP ranges (127.0.0.1, 10.x, 172.16-31.x,")
print("     192.168.x, and 169.254.x — which is where cloud metadata lives)")
print("  3. Domain allowlist (only fetch from approved domains)")
print("  4. DNS resolution check (resolve the hostname, verify it's not private)")
print("  5. allow_redirects=False — without it, an allowlisted host can answer")
print("     302 -> http://169.254.169.254/ and every check above is bypassed,")
print("     because requests would resolve and fetch the redirect target itself.")

for url, _, expect in test_cases:
    if expect == "blocked":
        assert results[url] in BLOCKED_BY_US, (
            f"{url} returned {results[url]} — the safe endpoint did NOT refuse a "
            f"URL it is supposed to refuse"
        )

# The allow case is the only one that needs the public internet. Don't let a
# flaky network turn into a false security result in either direction.
allowed_status = results["https://httpbin.org/get"]
assert allowed_status not in BLOCKED_BY_US, (
    f"the allowlisted domain httpbin.org was refused with {allowed_status} — the "
    f"allowlist is broken and the endpoint is useless, not merely safe"
)
if allowed_status != 200:
    print(f"\nℹ️  httpbin.org answered {allowed_status} (upstream/offline). Our checks")
    print("   let it through, which is the part this cell is testing.")

print("\n⚠️  HONEST LIMITATION: this is defence in depth, not a proof. We resolve")
print("   the hostname to check it, and `requests` resolves it again to fetch it.")
print("   An attacker who controls DNS can return a public IP for the first lookup")
print("   and 169.254.169.254 for the second (DNS rebinding). The production answer")
print("   is to resolve once, pin the validated IP, connect to that IP with an")
print("   explicit Host header — or route all outbound calls through an egress")
print("   proxy that enforces the allowlist where DNS can't move underneath you.")

---

## 🚨 Vulnerability 5: Missing Security Headers

**OWASP Category**: A05:2021 — Security Misconfiguration

### What Are Security Headers?

HTTP response headers act like **instructions to the browser**: "don't run scripts
from random places", "never load this page in an iframe", "always use HTTPS next
time you visit". Missing these headers doesn't create a single dramatic exploit,
but it removes several **layers of defense** that mitigate XSS, clickjacking,
protocol downgrades, and information leaks.

| Header | What it does | Protects against |
|--------|--------------|-------------------|
| `Content-Security-Policy` | Lists which sources of scripts/styles are allowed | XSS (defense in depth) |
| `Strict-Transport-Security` (HSTS) | Tell browser to always use HTTPS | Protocol downgrade / MITM |
| `X-Frame-Options` or CSP `frame-ancestors` | Prevent your page from being embedded | Clickjacking |
| `X-Content-Type-Options: nosniff` | Stop MIME-type guessing | Drive-by content execution |
| `Referrer-Policy` | Control how much URL info is sent to other sites | Info disclosure via Referer |
| `Permissions-Policy` | Disable browser APIs the page doesn't need | Misuse of mic/camera/geolocation |

Think of them as **free insurance** — cheap to add, and they dramatically reduce
the blast radius of any other bug you might ship.


In [ ]:
# === ATTACK 5: Missing headers means no browser-side safety net ===

print("🚨 Inspecting headers on the VULNERABLE page:\n")
resp = requests.get(f"{BASE_URL}/page/profile")
print(f"  Status: {resp.status_code}")
print("  Response headers:")
for k, v in resp.headers.items():
    print(f"    {k}: {v}")

expected = [
    "Content-Security-Policy",
    "Strict-Transport-Security",
    "X-Frame-Options",
    "X-Content-Type-Options",
    "Referrer-Policy",
    "Permissions-Policy",
]
missing = [h for h in expected if h not in resp.headers]
print(f"\n  ❌ Missing security headers: {len(missing)}/{len(expected)}")
for h in missing:
    print(f"     - {h}")

print("\n⚠️  With no CSP, any XSS bug executes attacker JavaScript freely.")
print("⚠️  With no X-Frame-Options, the page can be loaded inside an <iframe>")
print("    on evil.com — classic clickjacking.")

assert len(missing) == len(expected), (
    f"only {len(missing)} of {len(expected)} security headers are missing from the "
    f"vulnerable page — it is no longer the 'before' half of a before/after pair"
)

In [ ]:
# === FIX: The same page, served with security headers ===

print("✅ Inspecting headers on the SAFE page:\n")
resp = requests.get(f"{BASE_URL}/page/profile/safe")
print(f"  Status: {resp.status_code}")
print("  Response headers:")
for k, v in resp.headers.items():
    print(f"    {k}: {v}")

present = [h for h in expected if h in resp.headers]
print(f"\n  ✅ Security headers present: {len(present)}/{len(expected)}")

print("\nEach of these headers is a small change in Flask, but together they give")
print("the browser enough information to block entire classes of attacks.")

assert len(present) == len(expected), (
    f"the 'safe' page is missing {sorted(set(expected) - set(resp.headers))} — a "
    f"partial header set is a partial fix"
)
# CSP is the one that actually does work at runtime; check it says something.
csp = resp.headers["Content-Security-Policy"]
assert "default-src" in csp and "'unsafe-inline'" not in csp, (
    f"CSP is present but toothless: {csp!r}"
)
print("\n⚠️  Honest scope: headers are instructions to the *browser*. They do")
print("   nothing for a non-browser client, and HSTS only takes effect once the")
print("   page is actually served over HTTPS — over plain http://localhost the")
print("   browser ignores it entirely. They shrink the blast radius of a bug; they")
print("   never replace fixing the bug.")

### How Big Sites Use Security Headers

- **GitHub, Google, Stripe** all set strict `Content-Security-Policy` so that even
  if an XSS bug sneaks through, attacker JavaScript is blocked by the browser.
- **Banks and healthcare apps** set `Strict-Transport-Security` with a long
  `max-age` and `preload`, getting themselves added to browsers' built-in
  HTTPS-only list so users can **never** accidentally visit over HTTP.
- You can inspect any website's security headers at
  <https://securityheaders.com/> or with `curl -I https://example.com`.


---

## Summary: Vulnerable vs. Fixed Code

| Vulnerability | Vulnerable Pattern | Fixed Pattern |
|--------------|-------------------|---------------|
| **SQL Injection** | `f"SELECT * WHERE name = '{input}'"` | `cur.execute("SELECT * WHERE name = %s", (input,))` |
| **XSS** | `f"<div>{user_content}</div>"` | `f"<div>{escape(user_content)}</div>"` |
| **SSTI** | `render_template_string(html_built_from_user_input)` | `make_response(html)` — or pass user data as a template **variable**, never as template text |
| **CSRF** | No token check | Validate CSRF token + check Origin header |
| **SSRF** | `requests.get(user_url)` | Allowlist domains + block private IPs + HTTPS only + `allow_redirects=False` |
| **Missing Security Headers** | No `Content-Security-Policy`, `X-Frame-Options`, `HSTS` | Return a complete set of security headers on every response |

Notice that the first three rows are the same mistake wearing different hats:
**user input became part of a program** — a SQL statement, an HTML document, a Jinja
template. The fix is always the same shape too: pass the input as a *parameter* to a
parser that already knows where the code ends and the data begins.

## Microsoft SDL Requirements for These Issues

| SDL Requirement | What It Means |
|----------------|---------------|
| **Use approved libraries** | Use ORM (SQLAlchemy) or parameterized queries — never raw string SQL |
| **Encode output** | All user-supplied data must be HTML-encoded before rendering |
| **Anti-forgery tokens** | All state-changing operations must use CSRF tokens |
| **Validate all input** | Allowlist validation for URLs, reject unexpected patterns |
| **Static analysis** | Run SAST tools (Bandit for Python) to catch these automatically |

## 🔑 Key Takeaways

1. **Never trust user input** — always validate, sanitize, and escape
2. **SQL Injection** is prevented with parameterized queries — never concatenate strings into SQL
3. **XSS** is prevented by escaping HTML output — use `markupsafe.escape()` or template auto-escaping
4. **SSTI is not XSS** — escaping `< > &` does nothing to `{{ }}`. Never build a template out of user input; pass it in as a variable
5. **CSRF** is prevented with tokens and origin checks — every state-changing request needs a CSRF token
6. **SSRF** is prevented with URL allowlists and IP blocking — never fetch arbitrary URLs from user input
7. These are the **most common** vulnerabilities — OWASP Top 10 has been tracking them for 20+ years

## ➡️ Next: Notebook 3 — Secrets Management

Our app has another problem: **hardcoded secrets in the source code**. Let's fix that.